In [0]:
%sql
MERGE INTO assaia_catalog.silver.dClientes AS tgt
USING (
  SELECT
    id,
    nome_completo,
    cpf,
    CASE 
        WHEN lower(genero) = 'feminino' THEN 'F'
        WHEN lower(genero) = 'masculino' THEN 'M'
        ELSE 'OUTRO'
    END AS genero,
    _line,
    _fivetran_synced
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY id
             ORDER BY _fivetran_synced DESC, _line DESC
           ) AS rn
    FROM assaia_catalog.bronze.dClientes
    WHERE id IS NOT NULL
  ) x
  WHERE rn = 1
  AND length(cpf) = 11
) AS src
ON tgt.id = src.id
WHEN MATCHED THEN UPDATE SET
  tgt.nome_completo    = src.nome_completo,
  tgt.cpf              = src.cpf,
  tgt.genero           = src.genero,
  tgt._line            = src._line,
  tgt._fivetran_synced = src._fivetran_synced
WHEN NOT MATCHED THEN INSERT (
  id, nome_completo, cpf, genero, _line, _fivetran_synced
) VALUES (
  src.id, src.nome_completo, src.cpf, src.genero, src._line, src._fivetran_synced
);

In [0]:
%sql
MERGE INTO assaia_catalog.silver.dGerentes AS tgt
USING (
  SELECT
    id,
    nome_gerente,
    cpf,
    _line,
    _fivetran_synced
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY id
             ORDER BY _fivetran_synced DESC, _line DESC
           ) AS rn
    FROM assaia_catalog.bronze.dGerentes
    WHERE id IS NOT NULL
  ) x
  WHERE rn = 1
) AS src
ON tgt.id = src.id
WHEN MATCHED THEN UPDATE SET
  tgt.nome_gerente     = src.nome_gerente,
  tgt.cpf              = src.cpf,
  tgt._line            = src._line,
  tgt._fivetran_synced = src._fivetran_synced
WHEN NOT MATCHED THEN INSERT (
  id, nome_gerente, cpf, _line, _fivetran_synced
) VALUES (
  src.id, src.nome_gerente, src.cpf, src._line, src._fivetran_synced
);

In [0]:
%sql
MERGE INTO assaia_catalog.silver.dEnderecos AS tgt
USING (
  SELECT
    id,
    logradouro,
    bairro,
    cidade,
    estado,
    uf,
    pais,
    _line,
    _fivetran_synced
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY id
             ORDER BY _fivetran_synced DESC, _line DESC
           ) AS rn
    FROM assaia_catalog.bronze.dEnderecos
    WHERE id IS NOT NULL
  ) x
  WHERE rn = 1
) AS src
ON tgt.id = src.id
WHEN MATCHED THEN UPDATE SET
  tgt.logradouro       = src.logradouro,
  tgt.bairro           = src.bairro,
  tgt.cidade           = src.cidade,
  tgt.estado           = src.estado,
  tgt.uf               = src.uf,
  tgt.pais             = src.pais,
  tgt._line            = src._line,
  tgt._fivetran_synced = src._fivetran_synced
WHEN NOT MATCHED THEN INSERT (
  id, logradouro, bairro, cidade, estado, uf, pais, _line, _fivetran_synced
) VALUES (
  src.id, src.logradouro, src.bairro, src.cidade, src.estado, src.uf, src.pais, src._line, src._fivetran_synced
);

In [0]:
%sql
MERGE INTO assaia_catalog.silver.dProdutos AS tgt
USING (
  SELECT
    id,
    descricao,
    marca,
    categorias,
    sub_categoria,
    CAST(custo AS DECIMAL(18,2)) AS custo,
    CAST(preco_venda AS DECIMAL(18,2)) AS preco_venda,
    _line,
    _fivetran_synced
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY id
             ORDER BY _fivetran_synced DESC, _line DESC
           ) AS rn
    FROM assaia_catalog.bronze.dProdutos
    WHERE id IS NOT NULL
  ) x
  WHERE rn = 1
) AS src
ON tgt.id = src.id
WHEN MATCHED THEN UPDATE SET
  tgt.descricao        = src.descricao,
  tgt.marca            = src.marca,
  tgt.categorias       = src.categorias,
  tgt.sub_categoria    = src.sub_categoria,
  tgt.custo            = src.custo,
  tgt.preco_venda      = src.preco_venda,
  tgt._line            = src._line,
  tgt._fivetran_synced = src._fivetran_synced
WHEN NOT MATCHED THEN INSERT (
  id, descricao, marca, categorias, sub_categoria, custo, preco_venda, _line, _fivetran_synced
) VALUES (
  src.id, src.descricao, src.marca, src.categorias, src.sub_categoria, src.custo, src.preco_venda, src._line, src._fivetran_synced
);

In [0]:
%sql
MERGE INTO assaia_catalog.silver.dLojas AS tgt
USING (
  SELECT
    id,
    nome_unidade,
    endereco,
    id_gerente_geral,
    _line,
    _fivetran_synced
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY id
             ORDER BY _fivetran_synced DESC, _line DESC
           ) AS rn
    FROM assaia_catalog.bronze.dLojas
    WHERE id IS NOT NULL
      AND id_gerente_geral IS NOT NULL
  ) x
  WHERE rn = 1
) AS src
ON tgt.id = src.id
WHEN MATCHED THEN UPDATE SET
  tgt.nome_unidade     = src.nome_unidade,
  tgt.endereco         = src.endereco,
  tgt.id_gerente_geral = src.id_gerente_geral,
  tgt._line            = src._line,
  tgt._fivetran_synced = src._fivetran_synced
WHEN NOT MATCHED THEN INSERT (
  id, nome_unidade, endereco, id_gerente_geral, _line, _fivetran_synced
) VALUES (
  src.id, src.nome_unidade, src.endereco, src.id_gerente_geral, src._line, src._fivetran_synced
);

In [0]:
%sql
MERGE INTO assaia_catalog.silver.fVendas AS tgt
USING (
  SELECT
    id_cliente,
    id_produto,
    qtd,
    data_venda,
    forma_pagamento,
    id_loja,
    _line,
    _fivetran_synced
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY id_cliente, id_produto, id_loja, data_venda
             ORDER BY _fivetran_synced DESC, _line DESC
           ) AS rn
    FROM assaia_catalog.bronze.fVendas
    WHERE id_cliente IS NOT NULL
      AND id_produto IS NOT NULL
      AND id_loja IS NOT NULL
      AND data_venda IS NOT NULL
  ) x
  WHERE rn = 1
) AS src
ON  tgt.id_cliente = src.id_cliente
AND tgt.id_produto = src.id_produto
AND tgt.id_loja    = src.id_loja
AND tgt.data_venda = src.data_venda
WHEN MATCHED THEN UPDATE SET
  tgt.qtd              = src.qtd,
  tgt.forma_pagamento  = src.forma_pagamento,
  tgt._line            = src._line,
  tgt._fivetran_synced = src._fivetran_synced
WHEN NOT MATCHED THEN INSERT (
  id_cliente, id_produto, qtd, data_venda, forma_pagamento, id_loja, _line, _fivetran_synced
) VALUES (
  src.id_cliente, src.id_produto, src.qtd, src.data_venda, src.forma_pagamento, src.id_loja, src._line, src._fivetran_synced
);